In [ ]:
print("test")

In [ ]:
import torch
torch.arange(10).cumsum(dim=0)

In [ ]:
import xarray as xr

In [ ]:
%load_ext autoreload
%autoreload 2

import torch
import numpy as np
from matplotlib import pyplot as plt
from pathlib import Path
from tqdm.auto import tqdm
from typing import Any

from uqct.datasets.utils import get_dataset
from uqct.metrics import get_metrics
from uqct.ct import fbp, nll, Experiment, Tomogram, anscombe_transform, radon, sinogram_from_counts, poisson, sample_observations, circular_mask

import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
import astra

astra.test_CUDA()

In [ ]:
def rmse(prediction, target, circle_mask=False):
    """
    Computes the Root Mean Square Error (RMSE) between two images.
    Args:
        prediction (torch.Tensor): (..., H, W) Predicted image tensor.
        target (torch.Tensor): (..., H, W) Target image tensor.
        circle_mask (bool): If True, applies a circular mask to both images before computing RM
    """
    if circle_mask:
        mask = circular_mask(target.shape[-1], device=target.device)
        target *= mask
        prediction *= mask
    mse = torch.sqrt(torch.mean((target - prediction) ** 2, dim=(-2, -1)))
    return mse

def psnr(prediction, target, max_pixel=None, circle_mask=False):
    if circle_mask:
        mask = circular_mask(target.shape[-1], device=target.device)
        target *= mask
        prediction *= mask
    if max_pixel is None:
        max_pixel = torch.max(target)
    mse = torch.mean((target - prediction) ** 2, dim=(-2, -1))
    psnr = 10 * torch.log10(max_pixel ** 2 / mse)
    return psnr

In [ ]:
dataset = 'lung'
train_set, test_set = get_dataset(dataset, True)

images = torch.stack([
    test_set[i] for i in range(5)
]).to(device)  # shape: (5, 1, 128, 128)
images_lr = F.interpolate(images, size=(128, 128), mode="area").to(device)

images.shape, images_lr.shape

In [ ]:
len(test_set)

In [ ]:
size_bytes = images.element_size() * images.numel() / 5 * 100
size_mb = size_bytes / (1024 ** 2)
print(f"images size: {size_bytes} bytes ({size_mb:.2f} MB)")

In [ ]:
N_BINS_HR = images.shape[-1]
print(N_BINS_HR)
N_BINS_LR = images.shape[-1]

num_angles = 200
total_intensities = torch.tensor([1e5, 1e6, 1e7, 1e8, 1e9], device=device).reshape(5, 1, 1, 1)
intensities_hr = total_intensities / num_angles / N_BINS_HR
intensities = intensities_hr * 2
angles = torch.from_numpy(np.linspace(0, 180, num_angles, endpoint=False)).float().to(device)


sinogram = radon(images, angles)
# intensities_hr = torch.ones(num_angles, device=device).unsqueeze(-1) / num_angles * 1e
counts = sample_observations(images, intensities_hr, angles)

# ToDo: clip sinogram from counts everywhere, or modify sinogram_from_counts to not produce negative values?
measurements_sinogram = sinogram_from_counts(counts, intensities).clip(0, 128)


fig, axes = plt.subplots(3, len(images), figsize=(3 * len(images), 9))
for i in range(len(images)):
    print(f"min/max of image {i}: {images[i].min().item()}/{images[i].max().item()}")
    axes[0, i].imshow(images[i].cpu().squeeze(), cmap='gray')
    axes[0, i].set_title(f'Image {i}')
    axes[0, i].axis('off')

    axes[1, i].imshow(sinogram[i].cpu().squeeze().T, cmap='gray')
    axes[1, i].set_title(f'Sinogram {i}')
    axes[1, i].axis('off')

    print(f"Sinogram range: {sinogram[i].min().item()}/{sinogram[i].max().item()}")

    axes[2, i].imshow(measurements_sinogram[i].cpu().squeeze().T, cmap='gray')
    axes[2, i].set_title(f'Sinogram CT {i}')
    axes[2, i].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
measurements_sinogram[0].min(), measurements_sinogram[0].max(), sinogram[0].min(), sinogram[0].max()

In [ ]:
intensities.shape, intensities.expand(counts.shape).sum(dim=(-1, -2))

In [ ]:

def fbp_recon(counts, intensities, angles):
    """Simple FBP reconstruction from an Experiment object."""
    sinogram = sinogram_from_counts(counts, intensities)
    return fbp(sinogram, angles)

print(counts.shape, intensities.shape)

# _counts = counts.unsqueeze(1).unsqueeze(1)  # (B, 1, n_angles, n_detectors)
# _intensities = intensities.unsqueeze(0).unsqueeze(1).unsqueeze(1).unsqueeze(1)
fbp_samples = fbp_recon(counts, intensities, angles).clip(0,1)
print(fbp_samples.shape, counts.shape, intensities.shape)


fig, axes = plt.subplots(2, len(images), figsize=(15, 6))
for i in range(len(images_lr)):
    metrics = get_metrics(images_lr[i], fbp_samples[i].squeeze())
    print(f"sample {i}: PSNR={metrics['PSNR']}, min/max of FBP sample {i}: {fbp_samples[i].min().item()}/{fbp_samples[i].max().item()}")

    axes[0, i].imshow(images[i].cpu().squeeze(), cmap='gray')
    axes[0, i].set_title(f'Image {i}')
    axes[0, i].axis('off')

    axes[1, i].imshow(fbp_samples[i].cpu().squeeze(), cmap='gray')
    axes[1, i].set_title(f'FBP Sample {i}')
    axes[1, i].axis('off')

In [ ]:
# Total-Variation (TV) regularizer (isotropic)
def tv_loss(img: torch.Tensor, reduction: str = "mean") -> torch.Tensor:
    """
    img: (..., H, W) or (H, W)
    Returns:
      - if reduction == "mean" -> tensor with shape of batch dims (...),
        where last two spatial dims have been reduced by mean over H,W
      - if reduction == "sum"  -> same but summed over H,W
      - if reduction is None   -> full tensor of same shape as img with last two dims = H,W
    Works with arbitrary number of leading batch dimensions.
    """
    # Accept 2D (H,W) as special case and keep track to squeeze later
    squeezed = False
    if img.dim() == 2:
        img = img.unsqueeze(0)
        squeezed = True

    assert img.dim() >= 3, "tv_loss expects input with at least 2 spatial dims"

    # differences along height (last-2) and width (last-1)
    dx = img[..., 1:, :] - img[..., :-1, :]      # (..., H-1, W)
    dy = img[..., :, 1:] - img[..., :, :-1]      # (..., H, W-1)

    dx2 = dx * dx
    dy2 = dy * dy

    # pad dx2 to shape (..., H, W) by adding zero row at end along height
    # pad format: (pad_w_left, pad_w_right, pad_h_left, pad_h_right)
    dx2_p = torch.nn.functional.pad(dx2, (0, 0, 0, 1), value=0.0)
    # pad dy2 to shape (..., H, W) by adding zero column at end along width
    dy2_p = torch.nn.functional.pad(dy2, (0, 1, 0, 0), value=0.0)

    mag = torch.sqrt(dx2_p + dy2_p + 1e-8)  # (..., H, W)

    if reduction == "mean":
        out = mag.mean(dim=(-2, -1))
    elif reduction == "sum":
        out = mag.sum(dim=(-2, -1))
    elif reduction is None:
        out = mag
    else:
        raise ValueError(f"Unknown reduction: {reduction}")

    if squeezed:
        # if original input was 2D, return scalar or 1-element tensor accordingly
        return out.squeeze(0)

    return out

class IterativeRecon:
    def __init__(
        self,
        steps: int = 200,
        init_method: str = "fbp",
        use_sigmoid: bool = False,
        lr: float = 1e-2,
        prior: torch.Tensor | None = None,
        loss: str = "nll",
        tv_weight: float = 0.0,  # added TV weight
        device: torch.device = device,
    ):
        self.steps = steps
        self.init_method = init_method
        self.use_sigmoid = use_sigmoid
        self.lr = lr
        self._prior = prior
        self.loss = loss
        self.tv_weight = tv_weight
        self.device = device

    def _build_prior(self, counts, angles, intensities):
        if self.init_method == "fbp":
            sinogram = sinogram_from_counts(counts, intensities)
            prior = fbp(sinogram, angles)
        # elif self.init_method == "fbp_weighted":
        #     prior = fbp(counts, angles, intensities, weighted=True)
        elif self.init_method == "zeros":
            bs = counts.shape[0:-2]
            prior = torch.zeros((*bs, counts.shape[-1], counts.shape[-1]), device=counts.device)
        elif self.init_method == "const":
            bs = counts.shape[0:-2]
            prior = torch.ones((*bs, counts.shape[-1], counts.shape[-1]), device=counts.device) * 0.5
        elif self.init_method == "random":
            bs = counts.shape[0:-2]
            prior = torch.randn((*bs, counts.shape[-1], counts.shape[-1]), device=counts.device).clip(0, 1)
        elif self.init_method == "prior" and self._prior is not None:
            prior = self._prior.clone().to(counts.device)
        else:
            raise ValueError(f"Unknown init_method: {self.init_method}")
        return prior

    def __call__(self, counts: torch.Tensor, intensities: torch.Tensor, angles: torch.Tensor) -> torch.Tensor:
        """
        Perform iterative reconstruction given counts, intensities, and angles.
        Returns reconstructed image tensor (B, H, W).
        """
        prior_img = self._build_prior(counts, angles, intensities)
        tomogram = Tomogram(prior=prior_img.detach(), use_sigmoid=self.use_sigmoid, circle=True)

        optimizer = torch.optim.Adam(tomogram.parameters(), lr=self.lr)
        # if self.loss == "nll":
        #     loss_fn = nll
        # else:
        #     # mse_ct must be available in scope if using this branch
        #     loss_fn = lambda recon, meas, angs, alloc: mse_ct(recon, meas, angs, alloc, vst=anscombe_transform)

        circle_mask = circular_mask(prior_img.shape[-1], device=tomogram.image.device)

        it = tqdm(range(self.steps), desc="Iterative Reconstruction")
        for step in it:
            optimizer.zero_grad()

            loss = nll(tomogram(), counts, intensities, angles).mean(dim=(-1, -2)).sum()

            if self.tv_weight:
                _tv_loss = tv_loss(tomogram(), reduction="mean").mean()
                loss += self.tv_weight * _tv_loss

            loss.backward()
            optimizer.step()

            # circle mask and clamp
            with torch.no_grad():
                tomogram.image.clamp_(min=0.0, max=1.0)
                tomogram.image.mul_(circle_mask)

            it.set_postfix(loss=f"{loss.item():.10f}, tv={_tv_loss.item() if self.tv_weight else 0.0:.10f}")

        with torch.no_grad():
            recon = tomogram()

        return recon.detach()


# usage: create instance then call with counts/intensities/angles
iterative_model = IterativeRecon(steps=80, use_sigmoid=False, init_method="zeros", lr=1e-1, tv_weight=0.)

samples = iterative_model(counts, intensities, angles).clip(0,1)
# mask = circular_mask(samples.shape[-1], device=samples.device)
# sampes = torch.nn.Parameter(samples.clone())
# samples *= mask.unsqueeze(0)


fig, axes = plt.subplots(2, len(images_lr), figsize=(15, 6))
for i in range(len(images_lr)):
    metrics = get_metrics(images_lr[i].squeeze(), samples[i].squeeze())
    print(f"sample {i}: PSNR={metrics['PSNR']}, min/max of FBP sample {i}: {samples[i].min().item()}/{samples[i].max().item()}, min/max of GT image {i}: {images_lr[i].min().item()}/{images_lr[i].max().item()}")

    axes[0, i].imshow(images_lr[i].cpu().squeeze(), cmap='gray', vmin=0, vmax=1)
    axes[0, i].set_title(f'Image {i}')
    axes[0, i].axis('off')

    axes[1, i].imshow(samples[i].cpu().squeeze(), cmap='gray', vmin=0, vmax=1)
    axes[1, i].set_title(f'Iterative Sample {i}')
    axes[1, i].axis('off')

plt.show()


In [ ]:
from pathlib import Path
from uqct.training.unet import ( MIN_TOTAL_INTENSITY, MAX_TOTAL_INTENSITY, N_ANGLES, build_unet, sample_fbp_sparse)
from diffusers.models.unets.unet_2d import UNet2DModel


def load_unet(ckpt_path: Path, sparse: bool) -> UNet2DModel:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    unet = build_unet(sparse).to(device)  # type: ignore
    ckpt = torch.load(ckpt_path, map_location="cpu")
    sd = ckpt["unet"]
    # Handle potential _orig_mod prefix
    if any(k.startswith("_orig_mod.") for k in sd.keys()):
        sd = {k.replace("_orig_mod.", "", 1): v for k, v in sd.items()}
    unet.load_state_dict(sd, strict=True)
    print(
        f"Loaded checkpoint: epoch={ckpt.get('epoch','?')}, val_loss={ckpt.get('val_loss','?')}"
    )
    return unet  # type: ignore

# ckpt_path = '/mydata/chip/shared/checkpoints/uqct/unet_dense_128_lamino_0.pt'
ckpt_path = Path('/mydata/chip/shared/checkpoints/uqct/unet_dense/unet_dense_128_lung_0.pt')
if not ckpt_path.exists():
    ckpt_path = Path("../checkpoints/unet_dense/unet_dense_128_lung_0.pt")

# Load model
unet = load_unet(ckpt_path, sparse=False).to(device).eval()

In [ ]:
class UNetRecon:
    def __init__(self, unet: UNet2DModel):
        self.unet = unet
    
    def __call__(self, counts, intensities, angles):
        sinogram = sinogram_from_counts(counts, intensities)
        fbp_recon = fbp(sinogram, angles)

        total_intensities = intensities.expand(counts.shape).sum(dim=(-1, -2))
        print(total_intensities)

        x_in = (fbp_recon * 2.0 - 1.0)
        exposure_norm = (
            (total_intensities - MIN_TOTAL_INTENSITY) / (MAX_TOTAL_INTENSITY - MIN_TOTAL_INTENSITY) * 999
        )


        batch_dims = x_in.size()[:-2]
        img_shape = x_in.shape[-2:]

        x_in = x_in.view(-1, *img_shape)
        while exposure_norm.dim() < len(batch_dims):
            exposure_norm = exposure_norm.unsqueeze(0)

        exposure_norm = exposure_norm.expand(batch_dims).reshape(-1)

        y = unet(x_in.unsqueeze(1), timestep=exposure_norm, return_dict=False)[0]
        y = ((y + 1.0) / 2.0).clamp(0.0, 1.0)  # (B,128,128)

        y = y.view(*batch_dims, *img_shape)
        return y


unet_recon = UNetRecon(unet)
with torch.no_grad():
    preds_lr = unet_recon(counts, intensities, angles)
preds_lr.shape, counts.shape

In [ ]:
def plot_imgs(images, counts, intensities, angles, *predictions):

    sinogram = sinogram_from_counts(counts, intensities)
    fbp_lr = fbp(sinogram, angles)  
    mse = F.mse_loss(images, preds_lr)
    print(f"MSE: {mse}")

    fbp_metrics = []
    pred_metrics = []
    for i in range(len(images)):
        m_fbp = get_metrics(images[i], fbp_lr[i], normalize_range=True, constrained=True)
        m_pred = get_metrics(
            images[i], preds_lr[i], normalize_range=True, constrained=True
        )
        fbp_metrics.append(m_fbp)
        pred_metrics.append(m_pred)




    # I0_vals = I0_lr.detach().float().view(num_examples, -1).mean(dim=1).cpu().numpy()
    # total_intensity = intensities.sum(dim=(-1, -2)).squeeze().expand(len(images), 1).detach().float().cpu().numpy()
    total_intensity = intensities.expand(counts.shape).sum(dim=(-1, -2)).detach().float().cpu().numpy()


    # Plot: 3 rows (GT / FBP / Pred) × N columns, show PSNR & SSIM on each column
    _, axes = plt.subplots(2 + len(predictions), len(images), figsize=(2.2 * len(images), 6.0))
    if len(images) == 1:
        axes = np.asarray(axes).reshape(3, 1)

    for i in range(len(images)):
        # GT + overlay with angles and I0
        axes[0, i].imshow(images[i].squeeze().cpu(), cmap="gray", vmin=0.0, vmax=1.0)
        axes[0, i].axis("off")
        if i == 0:
            axes[0, i].set_title("GT", fontsize=10)

        # Overlay text (top-left)
        overlay_txt = f"{len(angles)} angles | I0={total_intensity[i].item():.3g}"
        axes[0, i].text(
            6,
            14,
            overlay_txt,
            color="white",
            fontsize=8,
            ha="left",
            va="top",
            bbox=dict(boxstyle="round,pad=0.2", fc="black", ec="none", alpha=0.6),
        )

        # FBP + metrics
        m_fbp = get_metrics(images[i], fbp_lr[i], normalize_range=True, constrained=True)
        axes[1, i].imshow(fbp_lr[i].squeeze().detach().cpu(), cmap="gray", vmin=0.0, vmax=1.0)
        axes[1, i].axis("off")
        axes[1, i].set_title(
            f"FBP  PSNR {m_fbp['PSNR']:.2f}  SSIM {m_fbp['SS']:.3f}",
            fontsize=8,
        )

        for j in range(len(predictions)):
            # Pred + metrics
            m_pred = get_metrics(
                images[i], predictions[j][i], normalize_range=True, constrained=True
            )
            axes[2 + j, i].imshow(predictions[j][i].squeeze().detach().cpu(), cmap="gray", vmin=0.0, vmax=1.0)
            axes[2, i].axis("off")
            axes[2, i].set_title(
                f"Pred PSNR {m_pred['PSNR']:.2f}  SSIM {m_pred['SS']:.3f}",
                fontsize=8,
            )

    plt.tight_layout()
    plt.show()


plot_imgs(images_lr, counts, intensities, angles, preds_lr)

In [ ]:
from uqct.models.diffusion import load_unet
from diffusers.models.unets.unet_2d import UNet2DModel
from diffusers.schedulers.scheduling_ddpm import DDPMScheduler
from uqct.models.guided_diffusion import GradientGuidance, GuidedDiffusionPipeline

In [ ]:
diffusion_unet = load_unet('/mydata/chip/shared/checkpoints/uqct/diffusion/ddpm_unconditional_128_lung.pt', cond=False)

In [ ]:
def guidance_loss(counts, intensities, angles, l=5.):
    """
    Define a loss function for the diffusion model.
    This can be used to guide the diffusion process.
    """
    data_shape = counts.shape[:-2]
    circle_mask = circular_mask(counts.shape[-1], device=counts.device)
    def loss_fn(image):
        img_shape = image.shape[-2:]
        image = image.view(-1, *data_shape, *img_shape)
        image = ((image + 1.0)/2).clip(0, 1)
        image = image * circle_mask
        loss =  nll(image, counts, intensities, angles, l=l)
        return loss.sum()
    return loss_fn


# scheduler = DDPMScheduler(num_train_timesteps=1000, beta_schedule="linear")
# guided_diffusion = GuidedDiffusionPipeline(diffusion_unet, scheduler)

# loss_fct = guidance_loss(counts, intensities, angles)
# guidance = GradientGuidance(
#     loss_fct=loss_fct, 
#     num_gradient_steps=5, 
#     guidance_start=1000, 
#     guidance_end=20, 
#     lr=1e-1
# )


# num_samples = 3
# samples = guided_diffusion(
#     batch_size=len(images) * num_samples,
#     num_inference_steps=100,
#     guidance=guidance,
#     verbose=True
# )
# samples = samples.view(num_samples, *counts.shape[:-2], 128, 128)

In [ ]:
class DiffusionRecon:

    def __init__(self, unet, scheduler, num_samples=5, num_inference_steps=100, timesteps=None, guidance_start=500, guidance_end=20, guidance_num_gradient_steps=10, guidance_lr=1e-1, guidance_lr_decay=False):
        self.unet = unet
        self.scheduler = scheduler
        self.num_samples = num_samples
        self.num_inference_steps = num_inference_steps
        self.timesteps = timesteps
        self.guidance_start = guidance_start
        self.guidance_end = guidance_end
        self.guidance_num_gradient_steps = guidance_num_gradient_steps
        self.guidance_lr = guidance_lr
        self.guidance_lr_decay = guidance_lr_decay

    def __call__(self, counts, intensities, angles, num_samples=None):
        if num_samples is None:
            num_samples = self.num_samples
        guided_diffusion = GuidedDiffusionPipeline(self.unet, self.scheduler)

        loss_fct = guidance_loss(counts, intensities, angles)
        guidance = GradientGuidance(
            loss_fct=loss_fct, 
            num_gradient_steps=self.guidance_num_gradient_steps,
            guidance_start=self.guidance_start,
            guidance_end=self.guidance_end, 
            lr=self.guidance_lr,
            learning_rate_decay=self.guidance_lr_decay
        )

        samples = guided_diffusion(
            batch_size=len(counts) * num_samples,
            num_inference_steps=self.num_inference_steps,
            timesteps=self.timesteps,
            guidance=guidance,
            verbose=True
        )
        return samples.view(num_samples, *counts.shape[:-2], 128, 128)


torch.manual_seed(1)

scheduler = DDPMScheduler(num_train_timesteps=1000, beta_schedule="linear")

num_inference_steps = 100
total_gradient_steps = 2000
num_gradient_steps = total_gradient_steps // num_inference_steps
print(num_inference_steps, num_gradient_steps)


diffusion_recon = DiffusionRecon(
    diffusion_unet, 
    scheduler, 
    num_samples=3, 
    num_inference_steps=num_inference_steps, 
    # timesteps=torch.linspace(500, 0, 50 + 1).int(),
    guidance_start=1000,
    guidance_end=10, 
    guidance_num_gradient_steps=num_gradient_steps, 
    guidance_lr=1e-1,
    guidance_lr_decay=True
)

samples = diffusion_recon(counts, intensities, angles, num_samples=3)
samples.shape

In [ ]:
plot_imgs(images_lr, counts, intensities, angles, samples.mean(dim=0))

In [ ]:
diffusion_cond_unet = load_unet('/mydata/chip/shared/checkpoints/uqct/diffusion/ddpm_conditional_128_lung.pt', cond=True)

In [ ]:
from uqct.training.unet import norm_intensities

class CondDiffusionRecon:

    def __init__(self, unet, scheduler, num_samples=5, num_inference_steps=100, timesteps=None, guidance_start=500, guidance_end=20, guidance_num_gradient_steps=10, guidance_lr=1e-1, guidance_lr_decay=False):
        self.unet = unet
        self.scheduler = scheduler
        self.num_samples = num_samples
        self.num_inference_steps = num_inference_steps
        self.timesteps = timesteps
        self.guidance_start = guidance_start
        self.guidance_end = guidance_end
        self.guidance_num_gradient_steps = guidance_num_gradient_steps
        self.guidance_lr = guidance_lr
        self.guidance_lr_decay = guidance_lr_decay

    def __call__(self, counts, intensities, angles, num_samples=None):
        if num_samples is None:
            num_samples = self.num_samples

        guided_diffusion = GuidedDiffusionPipeline(self.unet, self.scheduler)

        image_shape = (len(counts) * num_samples, 1, self.unet.config.sample_size, self.unet.config.sample_size)

        loss_fct = guidance_loss(counts, intensities, angles)
        guidance = GradientGuidance(
            loss_fct=loss_fct, 
            num_gradient_steps=self.guidance_num_gradient_steps,
            guidance_start=self.guidance_start,
            guidance_end=self.guidance_end, 
            lr=self.guidance_lr,
            learning_rate_decay=self.guidance_lr_decay
        )

        # compute fbps
        sinogram = sinogram_from_counts(counts, intensities)
        fbps = fbp(sinogram, angles)
        print(intensities.shape)
        intensities = intensities.expand(counts.shape).sum(dim=(-1, -2)).squeeze(0)

        # print(fbps.shape, counts.shape, intensities.shape)
        fbps_norm = (fbps * 2.0 - 1.0)
        fbps_norm = fbps_norm.unsqueeze(0).expand(num_samples, -1, -1, -1, -1).reshape(-1, 1, fbps.shape[-2], fbps.shape[-1])
        # print(fbps_norm.shape)
        

        # compute intensity
        print(total_intensities)

        intensities_norm = (2 * ((norm_intensities(intensities) / 999) - 0.5)).clip(
            -1, 1
        )
        print(intensities_norm.shape)
        intensities_norm = intensities_norm.unsqueeze(0).expand(num_samples, -1, -1).reshape(-1)
        print(intensities_norm.shape)

        print(angles.shape, N_ANGLES)
        n_angles = torch.tensor(len(angles), device=angles.device).float()
        n_angles_norm = ((n_angles - N_ANGLES / 2) / (N_ANGLES / 2)).clip(-1, 1)

        n_angles_norm = n_angles_norm.unsqueeze(0).unsqueeze(0).unsqueeze(0).expand(num_samples, len(counts), 1).reshape(-1)
        print(n_angles_norm.shape)

        cond_kwargs = dict(
            fbp=fbps_norm,
            intensity_norm=intensities_norm,
            n_angles_norm = n_angles_norm
        )

        print(fbps_norm.shape, intensities_norm.shape, n_angles_norm)

        samples = guided_diffusion(
            batch_size=len(counts) * num_samples,
            num_inference_steps=self.num_inference_steps,
            timesteps=self.timesteps,
            guidance=guidance,
            verbose=True,
            cond_kwargs=cond_kwargs,
            image_shape=image_shape
        )
        return samples.view(num_samples, *counts.shape[:-2], 128, 128)


torch.manual_seed(1)

scheduler = DDPMScheduler(num_train_timesteps=1000, beta_schedule="linear")

num_inference_steps = 100
total_gradient_steps = 2000
num_gradient_steps = total_gradient_steps // num_inference_steps
print(num_inference_steps, num_gradient_steps)

total_intensity = 1e7
beta_0 = -0.0786
beta_1 = 0.01
lr = beta_1 * np.log(total_intensity) + beta_0
print("Computed lr:", lr, "Total intensity:", total_intensity)

# lr=1e-1

diffusion_recon = CondDiffusionRecon(
    diffusion_cond_unet, 
    scheduler, 
    num_samples=3, 
    num_inference_steps=num_inference_steps, # 100
    guidance_start=1000,
    guidance_end=5, 
    guidance_num_gradient_steps=num_gradient_steps, # 20
    guidance_lr=1e-3,
    guidance_lr_decay=False
)

samples = diffusion_recon(counts, intensities, angles, num_samples=3)
samples.shape

In [ ]:
plot_imgs(images_lr, counts, intensities, angles, samples.mean(dim=0))

In [ ]:
def schedule_uniform(total_intensity, n_steps, init_fraction=None, device=None):
    if init_fraction:
        alloc = [init_fraction]
    else:
        init_fraction = 0.
        alloc = []
    per_step_fraction = (1. - init_fraction) / n_steps
    alloc += [per_step_fraction] * n_steps
    return torch.tensor(alloc, device=device) * total_intensity

def schedule_exponential(total_intensity, n_steps, init_fraction=None, base=2.0, device=None):
    if init_fraction:
        alloc = [init_fraction]
    else:
        init_fraction = 0.
        alloc = []
    exp_factors = torch.tensor([base ** i for i in range(n_steps)], device=device)
    exp_factors = exp_factors / exp_factors.sum()
    alloc += ((1. - init_fraction) * exp_factors).tolist()
    return torch.tensor(alloc, device=device) * total_intensity

schedule = schedule_uniform(1e6, n_steps=10, init_fraction=0.5, device=device)
schedule, schedule.sum()

schedule = schedule_exponential(1e6, n_steps=10, init_fraction=0., base=2.0, device=device)
schedule, schedule.sum(), len(schedule)

In [ ]:
schedule = schedule_uniform(1e6, n_steps=10, init_fraction=0.1, device=device)
schedule = schedule.reshape(1, -1, 1, 1, 1).expand(1, -1, 1, len(angles), 1) / len(angles)
data = sample_observations(images.unsqueeze(1), schedule / 2, angles) # TODO: divide by 2 is important!!!

sinograms = sinogram_from_counts(data, schedule)

In [ ]:
# for i in range(11):
#     plt.figure()
#     fbp_ = fbp(sinograms[0, i], angles)
#     plt.imshow(fbp_.squeeze().cpu().T, cmap='gray')
#     plt.show()

In [ ]:
print(data.shape, schedule.shape, angles.shape)
pred_fbp = fbp_recon(data.cumsum(dim=1), schedule.cumsum(dim=1), angles)

In [ ]:
_data = data.cumsum(dim=1)[:, -1]
_schedule = schedule.cumsum(dim=1)[:, -1]

with torch.no_grad():
    _preds = unet_recon(_data, _schedule, angles)
_preds

fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for i in range(5):
    axes[i].imshow(_preds[i].cpu().squeeze(), cmap='gray', vmin=0, vmax=1)
    axes[i].axis('off')
plt.show()

In [ ]:
with torch.no_grad():
    pred_unet = [unet_recon(data.cumsum(dim=1)[:, i], schedule.cumsum(dim=1)[:, i], angles) for i in range(data.shape[1])]
pred_unet = torch.stack(pred_unet, dim=1)
pred_unet.shape

fig, axes = plt.subplots(2, len(images), figsize=(15, 6))
for i in range(len(images)):
    axes[0, i].imshow(images[i].cpu().squeeze(), cmap='gray')
    axes[0, i].set_title(f'Image {i}')
    axes[0, i].axis('off')

    axes[1, i].imshow(pred_unet[i, -1].cpu().squeeze(), cmap='gray')
    axes[1, i].set_title(f'Recon Sample {i}')
    axes[1, i].axis('off')

plt.show()

In [ ]:
diffusion_recon = DiffusionRecon(
    diffusion_unet, 
    scheduler, 
    num_samples=3, 
    num_inference_steps=100, 
    guidance_start=1000, 
    guidance_end=10, 
    guidance_num_gradient_steps=10, 
    guidance_lr=1e-1
)
with torch.no_grad():
    pred_diffusion = [diffusion_recon(data.cumsum(dim=1)[:, i], schedule.cumsum(dim=1)[:, i], angles, num_samples=5).mean(dim=0) for i in range(data.shape[1])]

pred_diffusion = torch.stack(pred_diffusion, dim=1)
pred_diffusion.shape

In [ ]:
cond_diffusion_recon = CondDiffusionRecon(
    diffusion_cond_unet, 
    scheduler, 
    num_samples=3, 
    num_inference_steps=100, 
    guidance_start=1000, 
    guidance_end=10, 
    guidance_num_gradient_steps=20, 
    guidance_lr=1e-3
)
with torch.no_grad():
    pred_cond_diffusion = [cond_diffusion_recon(data.cumsum(dim=1)[:, i], schedule.cumsum(dim=1)[:, i], angles, num_samples=5).mean(dim=0) for i in range(data.shape[1])]

pred_cond_diffusion = torch.stack(pred_cond_diffusion, dim=1)
pred_cond_diffusion.shape

In [ ]:

for _pred in [pred_fbp, pred_unet, pred_diffusion, pred_cond_diffusion]:
    fig, axes = plt.subplots(2, len(images), figsize=(15, 6))
    for i in range(len(images)):
        axes[0, i].imshow(images[i].cpu().squeeze(), cmap='gray')
        axes[0, i].set_title(f'Image {i}')
        axes[0, i].axis('off')

        axes[1, i].imshow(_pred[i, -1].cpu().squeeze(), cmap='gray')
        axes[1, i].set_title(f'Recon Sample {i}')
        axes[1, i].axis('off')

    plt.show()

In [ ]:
for _pred in [pred_fbp, pred_unet, pred_diffusion, pred_cond_diffusion]:
    _psnr = psnr(_pred, images_lr.unsqueeze(1), circle_mask=False)
    _rmse = rmse(_pred, images_lr.unsqueeze(1), circle_mask=False)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    for i in range(len(images_lr)):
        # print(f"Sample {i}: PSNR = {_psnr[i].item():.2f} dB")
        ax1.plot(schedule.sum(-2).cumsum(dim=1).squeeze().cpu().numpy(), _psnr[i].squeeze().cpu().numpy(), label=f'Sample {i}')
        ax2.plot(schedule.sum(-2).cumsum(dim=1).squeeze().cpu().numpy(), _rmse[i].squeeze().cpu().numpy(), label=f'Sample {i}')
    
    ax1.legend()
    ax2.legend()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

for _pred, _name in [(pred_fbp, "FBP"), (pred_unet, "UNet"), (pred_diffusion, "Diffusion"), (pred_cond_diffusion, "CondDiffusion")]:
    _psnr = psnr(_pred, images_lr.unsqueeze(1), circle_mask=False).mean(dim=0)
    _rmse = rmse(_pred, images_lr.unsqueeze(1), circle_mask=False).mean(dim=0)

    ax1.plot(schedule.sum(-2).cumsum(dim=1).squeeze().cpu().numpy(), _psnr.squeeze().cpu().numpy(), label=_name)
    ax2.plot(schedule.sum(-2).cumsum(dim=1).squeeze().cpu().numpy(), _rmse.squeeze().cpu().numpy(), label=_name)

    ax1.legend()
    ax2.legend()

    ax1.set_ylabel("PSNR (dB)")
    ax2.set_ylabel("RMSE")
    ax1.set_xlabel("Total Intensity")
    ax2.set_xlabel("Total Intensity")


In [ ]:
for _pred in [pred_fbp, pred_unet]:
    _psnr = psnr(_pred, images.unsqueeze(1).unsqueeze(1), circle_mask=False)
    _rmse = rmse(_pred, images.unsqueeze(1).unsqueeze(1), circle_mask=False)
    

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    for i in range(len(images)):
        # print(f"Sample {i}: PSNR = {_psnr[i].item():.2f} dB")
        ax1.plot(schedule.sum(-2).cumsum(dim=1).squeeze().cpu().numpy(), _psnr[i].squeeze().cpu().numpy(), label=f'Sample {i}')
        ax2.plot(schedule.sum(-2).cumsum(dim=1).squeeze().cpu().numpy(), _rmse[i].squeeze().cpu().numpy(), label=f'Sample {i}')
    
    ax1.legend()
    ax2.legend()

In [ ]:
import torchvision.transforms.functional as TF

degree = 1

def rotate_images(images, degree):
    return torch.vmap(lambda img: TF.rotate(img, degree))(images)

rotated_images = rotate_images(images_lr, degree)
fig, axes = plt.subplots(2, len(images_lr), figsize=(15, 6))
for i in range(len(images_lr)):
    axes[0, i].imshow(images_lr[i].cpu().squeeze(), cmap='gray')
    axes[0, i].set_title(f'Image {i}')
    axes[0, i].axis('off')

    axes[1, i].imshow(rotated_images[i].cpu().squeeze(), cmap='gray')
    axes[1, i].set_title(f'Rotated Image {i}')
    axes[1, i].axis('off')

In [ ]:
images.shape, rotated_images.shape, images_lr.shape

In [ ]:
import torch.nn.functional as F
from uqct.debugging import plot_img
from uqct.ct import nll


predictions = {}
predictions['fbp'] = pred_fbp
predictions['unet'] = pred_unet 
predictions['diffusion'] = pred_diffusion
predictions['cond_diffusion'] = pred_cond_diffusion

nll_fbp = nll(predictions['fbp'][:, :-1].contiguous(), data[:, 1:], schedule[:, 1:], angles).sum((-1,-2))
nll_gt = nll(images_lr.unsqueeze(1).contiguous(), data[:, 1:], schedule[:, 1:], angles).sum((-1,-2))
nll_unet = nll(predictions['unet'][:, :-1].contiguous(), data[:, 1:], schedule[:, 1:], angles).sum((-1, -2))
nll_diffusion = nll(predictions['diffusion'][:, :-1].contiguous(), data[:, 1:], schedule[:, 1:], angles).sum((-1, -2))
nll_cond_diffusion = nll(predictions['cond_diffusion'][:, :-1].contiguous(), data[:, 1:], schedule[:, 1:], angles).sum((-1, -2))
nll_gt_rotated = nll(rotated_images.unsqueeze(1), data[:, 1:], schedule[:, 1:], angles).sum((-1,-2))


for i in range(len(images)):
    plt.figure()
    plt.plot(schedule.sum(-2).cumsum(dim=1)[0, 1:].squeeze().cpu(), (nll_fbp[i, :, 0] - nll_gt[i, :, 0]).cumsum(dim=0).cpu(), marker='o', label='FBP NLL')
    plt.plot(schedule.sum(-2).cumsum(dim=1)[0, 1:].squeeze().cpu(), (nll_unet[i, :, 0] - nll_gt[i, :, 0]).cumsum(dim=0).cpu(), marker='x', label='UNet NLL')
    # plt.plot(schedule.sum(-2).cumsum(dim=1)[0, 1:].squeeze().cpu(), (nll_diffusion[i, :, 0] - nll_gt[i, :, 0]).cumsum(dim=0).cpu(), marker='^', label='Diffusion NLL')
    plt.plot(schedule.sum(-2).cumsum(dim=1)[0, 1:].squeeze().cpu(), (nll_cond_diffusion[i, :, 0] - nll_gt[i, :, 0]).cumsum(dim=0).cpu(), marker='s', label='CondDiffusion NLL')
    plt.plot(schedule.sum(-2).cumsum(dim=1)[0, 1:].squeeze().cpu(), (nll_fbp[i, :, 0] - nll_gt_rotated[i, :, 0]).cumsum(dim=0).cpu(), marker='o', linestyle='--', label='FBP NLL')
    plt.plot(schedule.sum(-2).cumsum(dim=1)[0, 1:].squeeze().cpu(), (nll_unet[i, :, 0] - nll_gt_rotated[i, :, 0]).cumsum(dim=0).cpu(), marker='x', linestyle='--', label='UNet NLL')
    # plt.plot(schedule.sum(-2).cumsum(dim=1)[0, 1:].squeeze().cpu(), (nll_diffusion[i, :, 0] - nll_gt_rotated[i, :, 0]).cumsum(dim=0).cpu(), marker='^', linestyle='--', label='Diffusion NLL')
    plt.plot(schedule.sum(-2).cumsum(dim=1)[0, 1:].squeeze().cpu(), (nll_cond_diffusion[i, :, 0] - nll_gt_rotated[i, :, 0]).cumsum(dim=0).cpu(), marker='s', linestyle='--', label='CondDiffusion NLL')
    # plt.plot(schedule[0, 1:].cumsum(dim=0).squeeze().cpu(), (nll_gt[i, :, 0] - nll_gt[i, :, 0]).cumsum(dim=0).cpu(), marker='s', label='GT NLL')
    # plt.plot(schedule[0, 1:].cumsum(dim=0).squeeze().cpu(), (nll_gt_rotated[i, :, 0] - nll_gt[i, :, 0]).cumsum(dim=0).cpu(), marker='^', label='Rotated GT NLL')
    plt.axhline(y=0, color='black', linestyle='--', linewidth=1)

    plt.legend()
    plt.title(f'Sample {i} NLL vs Total Intensity')
    plt.xlabel('Total Intensity')
    plt.ylabel('beta_t')

    # plt.xscale('log')
    # plt.yscale('log')
    plt.show()



# plt.plot(schedule[:, 1:].cumsum(dim=0).cpu(), (nll_fbp[:, :, 0] - nll_gt[:, :, 0]).cumsum(dim=1).cpu(), marker='x', label='FBP NLL - GT NLL')
# plt.plot(schedule[:, 1:].cumsum(dim=0).cpu(), (nll_unet[:,0] - nll_gt[:, 0]).cumsum(dim=0).cpu(), marker='x', label='UNet NLL - GT NLL')
# plt.plot(schedule[:, 1:].cumsum(dim=0).cpu(), (nll_unet[:, 0] - nll_gt_rotated[:, 0]).cumsum(dim=0).cpu(), marker='x', label='UNet NLL - Rotated GT NLL')
# plt.plot(schedule[:, 1:].cumsum(dim=0).cpu(), (nll_fbp[:, 0] - nll_gt_rotated[:, 0]).cumsum(dim=0).cpu(), marker='x', label='FBP NLL - Rotated GT NLL')
# plt.axhline(y=0, color='black', linestyle='--', linewidth=1)

# plt.legend()

# plt.xscale('log')
# plt.yscale('log')


In [ ]:
nll_fbp[4] - nll_unet[4]

In [ ]:
# def rmse(predictions, gt):
#     return torch.sqrt(F.mse_loss(predictions, gt, reduction='none').mean(dim=(-1, -2)))

images = (
    F.interpolate(gt.unsqueeze(1), size=(128, 128), mode="area")
    .squeeze(1)
    .to(fbp_lr.device)
)  # match FBP/Pred resolution

print(images.shape, predictions['fbp'].shape)

plt.plot(schedule.cumsum(dim=0).cpu(), rmse(predictions['fbp'], images.unsqueeze(0)).mean(dim=-1).cpu(), label='FBP')
plt.plot(schedule.cumsum(dim=0).cpu(), rmse(predictions['unet'], images.unsqueeze(0)).mean(dim=-1).cpu(), label='unet')

plt.xscale('log')

In [ ]:
fig, axes = plt.subplots(len(schedule), 3, figsize=(9, 3 * len(schedule)))

for i, I_tot in enumerate(schedule):
    axes[i, 0].imshow(xs[0].cpu(), cmap='gray', vmin=0, vmax=1)
    axes[i, 0].set_title(f'GT')
    axes[i, 0].axis('off')

    axes[i, 1].imshow(predictions['fbp'][i, 0].cpu(), cmap='gray', vmin=0, vmax=1)
    axes[i, 1].set_title(f'FBP, I_tot={I_tot.item():.0f}')
    axes[i, 1].axis('off')

    axes[i, 2].imshow(predictions['unet'][i, 0].cpu(), cmap='gray', vmin=0, vmax=1)
    axes[i, 2].set_title(f'UNet, I_tot={I_tot.item():.0f}')
    axes[i, 2].axis('off')


In [ ]:
import matplotlib.pyplot as plt


mean_samples = samples.mean(axis=0)

_, axes = plt.subplots(num_samples + 2, len(images), figsize=(15, 3 * (num_samples + 2)))

for i in range(len(images)):
    axes[0, i].imshow(images[i].cpu().squeeze(), cmap='gray')
    axes[0, i].set_title(f'Image {i}')
    axes[0, i].axis('off')

    # fbp reconstruction
    axes[1, i].imshow(fbp_samples[i].cpu().squeeze(), cmap='gray')
    axes[1, i].set_title(f'FBP Reconstruction {i}')
    axes[1, i].axis('off')
    

    metrics = get_metrics(mean_samples[i].squeeze(), images[i])
    print(f"mean sample {i}: PSNR={metrics['PSNR']}, min/max of mean sample {i}: {mean_samples[i].min()}/{mean_samples[i].max()}")
    for j in range(num_samples):
        axes[j + 2, i].imshow(samples[j, i].cpu().squeeze(), cmap='gray')
        axes[j + 2, i].set_title(f'Sample {j} of Image {i}')
        axes[j + 2, i].axis('off')

In [ ]:
def guidance_loss(measurements, exposure, angles, batch_dims, img_shape, l=5.):
    """
    Define a loss function for the diffusion model.
    This can be used to guide the diffusion process.
    """
    def loss_fn(image):
        image = ((image + 1.0)/2).clip(0, 1)
        image = image.view(*batch_dims, *img_shape)
        loss =  nll(image, measurements, exposure, angles, l=l)
        return loss.mean(dim=(-1,-2)).sum()
    return loss_fn

def flatten_batch(images):
    batch_dims = images.size()[:-3]
    img_shape = images.size()[-3:]
    images = images.view(-1, *img_shape)
    return images, batch_dims, img_shape

def unflatten_batch(images, batch_dims, img_shape):
    images = images.view(*batch_dims, *img_shape)
    return images

def norm_image(image: torch.Tensor) -> torch.Tensor:
    return (image - 0.5) * 2


def denorm_image(
    image: torch.Tensor, min_v: float = 0.0, max_v: float = 1.0
) -> torch.Tensor:
    return ((image + 1) / 2).clip(min_v, max_v)

# @torch.inference_mode()
def generate_samples(
    unet: UNet2DModel,
    sample,
    noise_scheduler: DDPMScheduler,
    n_steps = None,
    loss_fct = None,
    n_grad_steps: int = 5,
    # measurements: torch.Tensor = None,
    # exposure: torch.Tensor = None,
    # angles: torch.Tensor = None,
) -> torch.Tensor:
    unet.eval()
    device = unet.device
    channels = unet.config["in_channels"]
    size = unet.config["sample_size"]
    sample, batch_dims, img_shape = flatten_batch(sample)

    if n_steps is None:
        n_steps = noise_scheduler.config["num_train_timesteps"]  # 1000
    noise_scheduler.set_timesteps(n_steps, device=device)

    circle_mask = get_circle_mask(img_shape[-1], device=device)

    for t in tqdm(
        noise_scheduler.timesteps,
        # total=len(noise_scheduler.timesteps),
        desc="denoising",
    ):
        # t is an int64 timestep compatible with the scheduler
        # Use autocast on CUDA for speed
        with torch.no_grad():
            with torch.autocast(
                device_type=device.type,
                dtype=torch.float16,
                enabled=(device.type == "cuda"),
            ):
                model_output = unet(sample, t, return_dict=False)[0]


        prev_t = scheduler.previous_timestep(t)

        if model_output.shape[1] == sample.shape[1] * 2 and scheduler.variance_type in ["learned", "learned_range"]:
            model_output, predicted_variance = torch.split(model_output, sample.shape[1], dim=1)
        else:
            predicted_variance = None

        # 1. compute alphas, betas
        alpha_prod_t = scheduler.alphas_cumprod[t]
        alpha_prod_t_prev = scheduler.alphas_cumprod[prev_t] if prev_t >= 0 else scheduler.one
        beta_prod_t = 1 - alpha_prod_t
        beta_prod_t_prev = 1 - alpha_prod_t_prev
        current_alpha_t = alpha_prod_t / alpha_prod_t_prev
        current_beta_t = 1 - current_alpha_t

        # 2. compute predicted original sample from predicted noise also called
        # "predicted x_0" of formula (15) from https://arxiv.org/pdf/2006.11239.pdf
        if scheduler.config.prediction_type == "epsilon":
            pred_original_sample = (sample - beta_prod_t ** (0.5) * model_output) / alpha_prod_t ** (0.5)
        elif scheduler.config.prediction_type == "sample":
            pred_original_sample = model_output
        elif scheduler.config.prediction_type == "v_prediction":
            pred_original_sample = (alpha_prod_t**0.5) * sample - (beta_prod_t**0.5) * model_output
        else:
            raise ValueError(
                f"prediction_type given as {self.config.prediction_type} must be one of `epsilon`, `sample` or"
                " `v_prediction`  for the DDPMScheduler."
            )

        # 3. Clip or threshold "predicted x_0"
        if scheduler.config.thresholding:
            pred_original_sample = scheduler._threshold_sample(pred_original_sample)
        elif scheduler.config.clip_sample:
            pred_original_sample = pred_original_sample.clamp(
                -scheduler.config.clip_sample_range, scheduler.config.clip_sample_range
            )

        # 4. Compute coefficients for pred_original_sample x_0 and current sample x_t
        # See formula (7) from https://arxiv.org/pdf/2006.11239.pdf
        pred_original_sample_coeff = (alpha_prod_t_prev ** (0.5) * current_beta_t) / beta_prod_t
        current_sample_coeff = current_alpha_t ** (0.5) * beta_prod_t_prev / beta_prod_t


        # guidance:
        x_0_pred = pred_original_sample
        # x_0_pred = denorm_image(x_0_pred)
        x_0_pred = x_0_pred.detach().clone().requires_grad_(True)

        # optimizer = torch.optim.SGD([x_0_pred], lr=1e-3)
        optimizer = torch.optim.Adam([x_0_pred], lr=1e-1)
        x_0_pred = unflatten_batch(x_0_pred, batch_dims, img_shape)

        if t > 10:
            it = tqdm(range(n_grad_steps), desc="Guidance Steps", leave=False, disable=True)
            for step in it:
                optimizer.zero_grad()
                loss = loss_fct(((x_0_pred + 1.0)/2).clip(0, 1)).sum()
                # loss = loss_fct(x_0_pred).sum()
                loss.backward()
                optimizer.step()
                it.set_postfix(loss=f"{loss.item():.10f}")

            #         # circle mask and clamp
            # with torch.no_grad():
            #     # tomogram.image.clamp_(min=0, max=1)
            #     x_0_pred.mul_(circle_mask)
        # with torch.no_grad():
        #     # tomogram.image.clamp_(min=0, max=1)
        #     # print(circle_mask[64, 64])
        #     x_0_pred.mul_(circle_mask)
        #     x_0_pred.add_(circle_mask - 1.)
        # x_0_pred = norm_image(x_0_pred.clip(0, 1))
        pred_original_sample = flatten_batch(x_0_pred)[0]
        # 5. Compute predicted previous sample µ_t
        # See formula (7) from https://arxiv.org/pdf/2006.11239.pdf
        pred_prev_sample = pred_original_sample_coeff * pred_original_sample + current_sample_coeff * sample

        # 6. Add noise
        variance = 0
        if t > 0:
            device = model_output.device
            # variance_noise = randn_tensor(
            #     model_output.shape, generator=generator, device=device, dtype=model_output.dtype
            # )
            variance_noise = torch.randn_like(model_output)
            if scheduler.variance_type == "fixed_small_log":
                variance = scheduler._get_variance(t, predicted_variance=predicted_variance) * variance_noise
            elif scheduler.variance_type == "learned_range":
                variance = scheduler._get_variance(t, predicted_variance=predicted_variance)
                variance = torch.exp(0.5 * variance) * variance_noise
            else:
                variance = (scheduler._get_variance(t, predicted_variance=predicted_variance) ** 0.5) * variance_noise

        pred_prev_sample = pred_prev_sample + variance


        sample = pred_prev_sample


        # out = noise_scheduler.step(model_output=noise_pred, timestep=t, sample=sample)

        # x_0_pred = out.pred_original_sample.requires_grad_(True)
        # noise = out.prev_sample - x_0_pred
        
        # loss_fct = guidance_loss(measurements, exposure, angles)
        # # optimizer = torch.optim.SGD([x_0_pred], lr=1e-3)
        # optimizer = torch.optim.Adam([x_0_pred], lr=1e-1)
        # x_0_pred = unflatten_batch(x_0_pred, batch_dims, img_shape)


        # if t > 10:
        #     it = tqdm(range(5), desc="Guidance Steps", leave=False, disable=True)
        #     for step in it:
        #         optimizer.zero_grad()
        #         loss = loss_fct(((x_0_pred + 1.0)/2).clip(0, 1)).sum()
        #         loss.backward()
        #         optimizer.step()
        #         it.set_postfix(loss=f"{loss.item():.10f}")

        #     #         # circle mask and clamp
        #     # with torch.no_grad():
        #     #     # tomogram.image.clamp_(min=0, max=1)
        #     #     x_0_pred.mul_(circle_mask)
        # # with torch.no_grad():
        # #     # tomogram.image.clamp_(min=0, max=1)
        # #     # print(circle_mask[64, 64])
        # #     x_0_pred.mul_(circle_mask)
        # #     x_0_pred.add_(circle_mask - 1.)
        
        # x_0_pred = flatten_batch(x_0_pred)[0]
        

        # variance = 0
        # predicted_variance = None
        # if t > 0:
        #     device = noise_pred.device
        #     # variance_noise = randn_tensor(
        #     #     noise_pred.shape, generator=generator, device=device, dtype=noise_pred.dtype
        #     # )
        #     variance_noise = torch.randn_like(noise_pred)
        #     if scheduler.variance_type == "fixed_small_log":
        #         variance = scheduler._get_variance(t, predicted_variance=predicted_variance) * variance_noise
        #     elif scheduler.variance_type == "learned_range":
        #         variance = scheduler._get_variance(t, predicted_variance=predicted_variance)
        #         variance = torch.exp(0.5 * variance) * variance_noise
        #     else:
        #         variance = (scheduler._get_variance(t, predicted_variance=predicted_variance) ** 0.5) * variance_noise
        # # sample = scheduler.add_noise(x_0_pred, torch.randn_like(sample), t)
        #     sample = x_0_pred + variance
        #     # sample = x_0_pred + noise
    sample = sample.detach()
    sample = unflatten_batch(sample, batch_dims, img_shape)

    x_0_pred = unflatten_batch(x_0_pred, batch_dims, img_shape)
    return x_0_pred



In [ ]:
counts.shape, intensities.shape, images.shape

In [ ]:
num_samples = 3
torch.manual_seed(1)
sample = torch.randn_like(
    images.unsqueeze(-3).unsqueeze(-3).repeat(1, num_samples, 1, 1, 1),
    device=unet.device,
    dtype=next(unet.parameters()).dtype,
)
sample.shape

In [ ]:
loss_fct = guidance_loss(counts.unsqueeze(-3), intensities, angles)

samples = generate_samples(unet, sample, scheduler, n_steps=100, loss_fct=loss_fct, n_grad_steps=2).detach()
samples = ((samples + 1.0) / 2).clip(0.0, 1.0)

mean_samples = samples.mean(axis=1)


import matplotlib.pyplot as plt

_, axes = plt.subplots(num_samples + 2, len(images), figsize=(15, 3 * (num_samples + 2)))

for i in range(len(images)):
    axes[0, i].imshow(images[i].cpu().squeeze(), cmap='gray')
    axes[0, i].set_title(f'Image {i}')
    axes[0, i].axis('off')

    # fbp reconstruction
    axes[1, i].imshow(fbp_samples[i].cpu().squeeze(), cmap='gray')
    axes[1, i].set_title(f'FBP Reconstruction {i}')
    axes[1, i].axis('off')
    

    metrics = get_metrics(mean_samples[i].squeeze(), images[i])
    print(f"mean sample {i}: PSNR={metrics['PSNR']}, min/max of mean sample {i}: {mean_samples[i].min()}/{mean_samples[i].max()}")
    for j in range(num_samples):
        axes[j + 2, i].imshow(samples[i, j].cpu().squeeze(), cmap='gray')
        axes[j + 2, i].set_title(f'Sample {j} of Image {i}')
        axes[j + 2, i].axis('off')

In [ ]:

samples = generate_samples(unet, sample, scheduler, n_steps=100, measurements=counts.unsqueeze(-3), exposure=intensities, angles=angles).detach()
samples = ((samples + 1.0) / 2).clip(0.0, 1.0)

mean_samples = samples.mean(axis=1)


import matplotlib.pyplot as plt

_, axes = plt.subplots(num_samples + 2, len(images), figsize=(15, 3 * (num_samples + 2)))

for i in range(len(images)):
    axes[0, i].imshow(images[i].cpu().squeeze(), cmap='gray')
    axes[0, i].set_title(f'Image {i}')
    axes[0, i].axis('off')

    # fbp reconstruction
    axes[1, i].imshow(fbp_samples[i].cpu().squeeze(), cmap='gray')
    axes[1, i].set_title(f'FBP Reconstruction {i}')
    axes[1, i].axis('off')
    

    metrics = get_metrics(mean_samples[i].squeeze(), images[i])
    print(f"mean sample {i}: PSNR={metrics['PSNR']}, min/max of mean sample {i}: {mean_samples[i].min()}/{mean_samples[i].max()}")
    for j in range(num_samples):
        axes[j + 2, i].imshow(samples[i, j].cpu().squeeze(), cmap='gray')
        axes[j + 2, i].set_title(f'Sample {j} of Image {i}')
        axes[j + 2, i].axis('off')

In [ ]:
mean_samples.shape, samples.shape

In [ ]:
from typing import Callable, Literal
from torch import optim

class Diffusion:
    def __init__(
        self,
        unet,
        num_steps: int = 50,
        buffer: int = 20,
        sgd_steps: int = 100,
        lr: float = 0.001,
        batch_size: int = 64,
        verbose: bool = False,
    ):
        self.verbose = verbose
        self.device = (
            torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
        )

        # U-Net
        self.batch_size = batch_size
        self.unet = unet
        self.unet.eval()
        for param in self.unet.parameters():  # type: ignore
            param.requires_grad = False

        # Diffusion
        self.num_steps = num_steps
        self.buffer = buffer
        self.noise_scheduler = DDPMScheduler(
            num_train_timesteps=1000, clip_sample=False
        )

        # Guidance
        self.sgd_steps = sgd_steps
        self.lr = lr

    def predict_x_0(
        self, t: int, x_t: torch.Tensor
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        device = x_t.device
        timesteps = torch.LongTensor([t]).to(device)

        noise_preds, x0_preds, x_t_prevs = [], [], []
        in_shape = x_t.shape
        x_t_flat = x_t.view(-1, 1, x_t.shape[-1], x_t.shape[-1])

        # Split into batches of size <= self.batch_size
        for batch in x_t_flat.split(self.batch_size):
            with torch.inference_mode():
                with torch.autocast(
                    device_type=device.type,
                    dtype=torch.float16,
                    enabled=(device.type == "cuda"),
                ):
                    noise_pred = self.unet(
                        batch,
                        timesteps,
                        return_dict=False,
                    )[0]

            step_result = self.noise_scheduler.step(
                noise_pred, int(timesteps.item()), batch.reshape(noise_pred.shape)
            )
            x_0_pred = step_result.pred_original_sample  # type: ignore
            x_t_previous = step_result.prev_sample  # type: ignore

            noise_preds.append(noise_pred)
            x0_preds.append(x_0_pred)
            x_t_prevs.append(x_t_previous)

        # Concatenate all batch results
        noise_pred = torch.cat(noise_preds, dim=0).view(in_shape)
        x_0_pred = torch.cat(x0_preds, dim=0).view(in_shape)
        x_t_previous = torch.cat(x_t_prevs, dim=0).view(in_shape)

        return noise_pred, x_0_pred, x_t_previous

    def step(self, t: torch.Tensor, target_t: int, x_t: torch.Tensor):
        device = x_t.device
        noise_pred, x_0_pred, _ = self.predict_x_0(int(t.item()), x_t)
        new_timestep = torch.LongTensor([target_t]).to(device)
        new_x_t = self.noise_scheduler.add_noise(
            x_0_pred,
            torch.randn_like(x_0_pred),
            new_timestep,  # type: ignore
        ).to(device)
        return new_x_t, noise_pred

    def reverse(
        self,
        x_t_start: torch.Tensor,
        t_start: int,
        t_end: int,
        num_steps=50,
    ) -> torch.Tensor:
        with torch.no_grad():
            x_t = x_t_start.clone()
            timesteps = torch.linspace(t_start, t_end, num_steps + 1).int()
            for i in tqdm(range(1, len(timesteps)), disable=not self.verbose):
                t = timesteps[i - 1]
                target_t = timesteps[i]
                x_t, _ = self.step(t, int(target_t.item()), x_t)
            return x_t

    def sample(
        self,
        x_t: torch.Tensor,
        # experiment: Experiment,
        guidance_loss_fn: Callable | None = None,
        l2_ball_radius: float = float("inf"),
    ) -> torch.Tensor:
        """
        Returns
            torch.Tensor: `(..., n_angles, replicates, 1, side_length, side_length)` (sparse) or `(..., n_rounds, replicates, 1, side_length, side_length)` (dense).
        """
        # TODO: Try restricting guidance output to a l2-ball around the current mean.

        # side_length = experiment.counts.shape[-1]
        # if experiment.sparse:
        #     # (rep, ..., n_angles, 1, side_length, side_length)
        #     x_t = torch.randn(
        #         replicates,
        #         *experiment.batch_dims,
        #         experiment.counts.shape[-2],
        #         side_length,
        #         side_length,
        #         device=self.device,
        #     )
        # else:
        #     x_t = torch.randn(
        #         replicates,
        #         *experiment.batch_dims,
        #         experiment.counts.shape[-3],
        #         side_length,
        #         side_length,
        #         device=self.device,
        #     )
        rep_first_shape = x_t.shape

        timesteps = torch.linspace(0, 500, self.num_steps + 1).int()
        it = tqdm(range(self.num_steps), disable=not self.verbose)
        optimizer = None

        for i in it:
            t = timesteps[-i - 1]
            target_t = timesteps[-i - 2]
            new_timestep = torch.LongTensor([target_t]).to(self.device)
            self.noise_scheduler.previous_timestep = lambda _: target_t  # type: ignore

            _, x_0_pred, _ = self.predict_x_0(int(t.item()), x_t)
            x_t = x_0_pred.view(rep_first_shape)

            if guidance_loss_fn is not None:
                lr = self.lr
                if self.num_steps - i < 20:
                    lr = self.lr * (self.num_steps - i) / 25
                x_t, guidance_loss, optimizer = guide(
                    x_t,
                    guidance_loss_fn,
                    sgd_steps=self.sgd_steps,
                    lr=lr,
                    optimizer=optimizer,
                    l2_ball_radius=l2_ball_radius,
                    verbose=False,
                )
                it.set_postfix({"loss": f"{guidance_loss:.3f}"})

            x_t = self.noise_scheduler.add_noise(
                x_t,
                torch.randn_like(x_t),
                new_timestep,  # type: ignore
            )

        x_t = self.reverse(x_t, self.buffer, 0, num_steps=self.buffer)
        _, x_t, _ = self.predict_x_0(0, x_t)
        out = denorm_image(x_t).reshape(rep_first_shape)

        # Massage from
        #    (replicates, ..., n_angles or n_rounds, side_length, side_length)
        # to (..., n_angles or n_rounds, replicates, 1, side_length, side_length),
        return out
        n_batch_dims = len(experiment.batch_dims)
        out_perm = (
            *tuple(range(1, n_batch_dims + 1)),
            n_batch_dims + 1,
            0,
            x_t.ndim - 2,
            x_t.ndim - 1,
        )
        return out.permute(out_perm).unsqueeze(-3)
    



def guide(
    x_t: torch.Tensor,
    loss_fct: Callable[..., torch.Tensor],
    sgd_steps: int = 50,
    lr: float = 0.1,
    optimizer: None | optim.Adam = None,
    l2_ball_radius: float = float("inf"),
    verbose: bool = False,
) -> tuple[torch.Tensor, float, optim.Adam]:
    circle_mask = torch.ones(*x_t.shape[-2:], device=x_t.device)
    radius = x_t.shape[-1] // 2
    y, x = torch.meshgrid(
        torch.arange(x_t.shape[-2], device=x_t.device),
        torch.arange(x_t.shape[-1], device=x_t.device),
        indexing="ij",
    )
    mask = (x - radius) ** 2 + (y - radius) ** 2 <= radius**2
    circle_mask[~mask] = 0
    loss = torch.tensor([float("inf")])

    y_0 = denorm_image(x_t)

    if optimizer is None:
        y = torch.nn.Parameter(y_0)
        optimizer = torch.optim.Adam([y], lr=lr)
    else:
        y = optimizer.param_groups[0]["params"][0]
        with torch.no_grad():
            y.copy_(y_0)
    it = tqdm(range(sgd_steps), disable=not verbose)
    for _ in it:
        optimizer.zero_grad()
        yp = y * mask
        loss = loss_fct(yp.clip(0))

        # Punish out of range
        low = yp[yp < 0]
        if len(low) > 0:
            loss += low.abs().mean()
        high = yp[yp > 1]
        if len(high) > 0:
            loss += (high - 1).abs().mean()

        loss.backward()
        optimizer.step()
        with torch.no_grad():
            y.data = project_to_l2_ball(y, y_0, l2_ball_radius)
            y.data[..., ~mask] = 0.0

        it.set_postfix({"loss": f"{loss.item():.3f}"})
    optimizer.zero_grad(set_to_none=True)
    x_t_guided = norm_image(y).clip(-1, 1)
    return x_t_guided, loss.item(), optimizer


def norm_image(image: torch.Tensor) -> torch.Tensor:
    return (image - 0.5) * 2


def denorm_image(
    image: torch.Tensor, min_v: float = 0.0, max_v: float = 1.0
) -> torch.Tensor:
    return ((image + 1) / 2).clip(min_v, max_v)


def project_to_l2_ball(
    y: torch.Tensor,
    y_0: torch.Tensor,
    l2_ball_radius: float | torch.Tensor,
    eps: float = 1e-12,
) -> torch.Tensor:
    """
    Project y onto the L2 ball of radius l2_ball_radius centered at y_0,
    where each sample is a (r, r) block in the last three dimensions.

    Shapes:
        y, y_0: (..., r, r)  (same shape, broadcast okay if compatible)
        l2_ball_radius: float or tensor broadcastable to (..., 1, 1)

    Returns:
        y_proj with the same shape as y.
    """
    diff = y - y_0
    norms = torch.linalg.vector_norm(diff, ord=2, dim=(-2, -1), keepdims=True)
    scale = torch.clamp(l2_ball_radius / (norms + eps), max=1.0)
    y_proj = y_0 + diff * scale
    return y_proj



In [ ]:
def get_guidance_loss_fn(
    counts, intensities, angles
) -> Callable[[torch.Tensor], torch.Tensor]:
    """Create a loss function that takes as input a batch of images and returns the Poisson NLL loss."""


    def loss_fn(image: torch.Tensor) -> torch.Tensor:
        return nll(image, counts, intensities, angles).mean()

    return loss_fn


class DiffusionRecon2:
    def __init__(self, unet):
        self.unet = unet

    def __call__(self, counts, intensities, angles, num_samples=3):

        diffusion = Diffusion(
            self.unet,
            num_steps=50,
            buffer=10,
            sgd_steps=100,
            lr=0.001,
            verbose=True,
        )
        img_shape = (counts.shape[-1], counts.shape[-1])
        batch_dims = counts.shape[:-2]

        guidance_loss_fn = get_guidance_loss_fn(counts, intensities, angles)
        x_t = torch.randn((num_samples, *batch_dims, *img_shape), device=counts.device)
        sample = diffusion.sample(x_t, guidance_loss_fn, 256 * 0.1)

        return sample  # (num_samples, ..., 1, H, W)
diffusion_recon2 = DiffusionRecon2(unet)
samples = diffusion_recon2(counts, intensities, angles, num_samples=3)


In [ ]:
images_lr.shape, counts.shape, intensities.shape, samples.shape

In [ ]:
sample.shape
mean_samples = samples.mean(axis=0)

plot_imgs(images_lr, counts, intensities, angles, mean_samples)